In [ ]:
import os
import json

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_credentials = {
    "username": "miftajessica",
    "key": "KGAT_2316b73dac820c84cfb14698e1294133"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Credentials saved!")

!kaggle datasets download -d banuprasadb/visdrone-dataset
!unzip -q visdrone-dataset.zip -d /content/visdrone
print("Done!")

for item in os.listdir('/content/visdrone'):
    print(item)

In [ ]:
import os

base = '/content/visdrone/VisDrone_Dataset'

# Full structure
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        subindent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            print(f'{subindent}{file}')
        if len(files) > 3:
            print(f'{subindent}... ({len(files)} files total)')

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

base = '/content/visdrone/VisDrone_Dataset'
train_images = base + '/VisDrone2019-DET-train/images'
train_labels = base + '/VisDrone2019-DET-train/labels'

# VisDrone class names
CLASS_NAMES = {
    0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car',
    4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle',
    8: 'bus', 9: 'motor'
}

# Classes we care about
HUMAN_CLASSES = [0, 1]  # pedestrian, people
CAR_CLASSES = [2, 3, 4, 5, 6, 7, 8, 9]  # all vehicles including car

# ── 1. Dataset Summary ──────────────────────────────────────────────
train_imgs = len(os.listdir(train_images))
val_imgs   = len(os.listdir(base + '/VisDrone2019-DET-val/images'))
test_imgs  = len(os.listdir(base + '/VisDrone2019-DET-test-dev/images'))

print("=" * 45)
print("        VISDRONE DATASET SUMMARY")
print("=" * 45)
print(f"  Train images : {train_imgs}")
print(f"  Val images   : {val_imgs}")
print(f"  Test images  : {test_imgs}")
print(f"  Total        : {train_imgs + val_imgs + test_imgs}")
print("=" * 45)

# ── 2. Class Distribution ────────────────────────────────────────────
class_counts = {i: 0 for i in range(10)}

for label_file in os.listdir(train_labels):
    with open(os.path.join(train_labels, label_file), 'r') as f:
        for line in f.readlines():
            cls = int(line.strip().split()[0])
            if cls in class_counts:
                class_counts[cls] += 1

print("\nClass Distribution (Train):")
for cls_id, count in class_counts.items():
    print(f"  {CLASS_NAMES[cls_id]:<20} : {count:,}")

# ── 3. Visualize Sample Images with Bounding Boxes ───────────────────
def draw_boxes(img_path, label_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    human_count = 0
    car_count = 0

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                cls = int(parts[0])
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)

                if cls in HUMAN_CLASSES:
                    color = (255, 0, 0)  # red for humans
                    human_count += 1
                elif cls in CAR_CLASSES:
                    color = (0, 255, 0)  # green for cars
                    car_count += 1
                else:
                    continue

                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

    return img, human_count, car_count

# Plot 6 sample images
img_files = sorted(os.listdir(train_images))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('VisDrone Sample Images\nRed = Humans | Green = Cars/Vehicles',
             fontsize=14, fontweight='bold')

for idx, (ax, img_file) in enumerate(zip(axes.flatten(), img_files)):
    img_path   = os.path.join(train_images, img_file)
    label_path = os.path.join(train_labels, img_file.replace('.jpg', '.txt'))

    img, h_count, c_count = draw_boxes(img_path, label_path)
    ax.imshow(img)
    ax.set_title(f'Humans: {h_count} | Cars: {c_count}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSample visualization saved!")

# ── 4. Class Distribution Bar Chart ─────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
classes = [CLASS_NAMES[i] for i in range(10)]
counts  = [class_counts[i] for i in range(10)]
colors  = ['#e74c3c' if i in HUMAN_CLASSES else '#2ecc71' for i in range(10)]

bars = ax.bar(classes, counts, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Class Distribution in Training Set\n(Red = Human Classes | Green = Vehicle Classes)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Class', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
plt.xticks(rotation=30, ha='right')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Class distribution chart saved!")

In [ ]:
# ── Task 02: Fine-tune YOLOv8 on VisDrone ───────────────────────────

!pip install -q ultralytics

import yaml
from ultralytics import YOLO

# Define the dataset configuration
dataset_config = {
    'path': '/content/visdrone/VisDrone_Dataset',
    'train': 'VisDrone2019-DET-train/images',
    'val': 'VisDrone2019-DET-val/images',
    'test': 'VisDrone2019-DET-test-dev/images',
    'nc': 10,
    'names': {
        0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car',
        4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle',
        8: 'bus', 9: 'motor'
    }
}

# Save the dataset configuration to visdrone.yaml
with open('/content/visdrone.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("visdrone.yaml created/updated!")

# Check the yaml file
with open('/content/visdrone.yaml', 'r') as f:
    print(yaml.safe_load(f))

In [ ]:
import os, json

# Step 1: Reinstall
!pip install -q ultralytics

# Step 2: Kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
kaggle_credentials = {
    "username": "miftajessica",
    "key": "KGAT_2316b73dac820c84cfb14698e1294133"
}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Step 3: Download dataset
!kaggle datasets download -d banuprasadb/visdrone-dataset
!unzip -q visdrone-dataset.zip -d /content/visdrone

# Step 4: Verify
val_path = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-val/images'
train_path = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/images'
print("Train exists:", os.path.exists(train_path))
print("Val exists:", os.path.exists(val_path))

In [ ]:
import yaml
from ultralytics import YOLO

# Save corrected yaml
dataset_config = {
    'path': '/content/visdrone/VisDrone_Dataset',
    'train': 'VisDrone2019-DET-train/images',
    'val': 'VisDrone2019-DET-val/images',
    'test': 'VisDrone2019-DET-test-dev/images',
    'nc': 10,
    'names': {
        0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car',
        4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle',
        8: 'bus', 9: 'motor'
    }
}

with open('/content/visdrone.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("YAML saved!")

# Load and train
model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/visdrone.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    name='visdrone_yolov8',
    project='/content/runs',
    device=0,
    patience=5,
    workers=2,
    verbose=True
)

print("\nTraining complete!")
print(f"Best model: {results.save_dir}")

In [ ]:
# Print training results
print(f"Training saved at: {results.save_dir}")

# Show training metrics
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f"{results.save_dir}/results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
print(df.tail(5))

# Plot training curves
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('YOLOv8 Training Results on VisDrone', fontsize=14, fontweight='bold')

metrics = [
    ('train/box_loss', 'Box Loss (Train)'),
    ('train/cls_loss', 'Class Loss (Train)'),
    ('metrics/mAP50(B)', 'mAP@50'),
    ('metrics/mAP50-95(B)', 'mAP@50-95'),
    ('metrics/precision(B)', 'Precision'),
    ('metrics/recall(B)', 'Recall')
]

for ax, (col, title) in zip(axes.flatten(), metrics):
    if col in df.columns:
        ax.plot(df[col], color='#e74c3c', linewidth=2)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nTraining curves saved!")

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import os

# Load best trained model
best_model_path = f"{results.save_dir}/weights/best.pt"
model = YOLO(best_model_path)
print(f"Model loaded: {best_model_path}")

# Test on sample images
test_images_path = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-val/images'
sample_images = sorted(os.listdir(test_images_path))[:6]

HUMAN_CLASSES = [0, 1]
CAR_CLASSES = [2, 3, 4, 5, 6, 7, 8, 9]

CLASS_NAMES = {
    0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car',
    4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle',
    8: 'bus', 9: 'motor'
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Task 03: Human & Car Detection with Counting\nRed = Humans | Green = Vehicles',
             fontsize=14, fontweight='bold')

for ax, img_file in zip(axes.flatten(), sample_images):
    img_path = os.path.join(test_images_path, img_file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # Run inference
    result = model(img_path, conf=0.25, verbose=False)[0]

    human_count = 0
    car_count = 0

    for box in result.boxes:
        cls = int(box.cls)
        conf = float(box.conf)
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if cls in HUMAN_CLASSES:
            color = (255, 0, 0)
            human_count += 1
        elif cls in CAR_CLASSES:
            color = (0, 200, 0)
            car_count += 1
        else:
            continue

        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        label = f"{CLASS_NAMES[cls]} {conf:.2f}"
        cv2.putText(img_rgb, label, (x1, max(y1-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    # Count overlay
    overlay_text = f"Humans: {human_count} | Vehicles: {car_count}"
    cv2.rectangle(img_rgb, (0, 0), (w, 30), (0, 0, 0), -1)
    cv2.putText(img_rgb, overlay_text, (10, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    ax.imshow(img_rgb)
    ax.set_title(f'Humans: {human_count} | Vehicles: {car_count}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/detection_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Detection results saved!")

In [ ]:
import cv2
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np

# Load best model
model = YOLO(f"{results.save_dir}/weights/best.pt")

HUMAN_CLASSES = [0, 1]
CAR_CLASSES = [2, 3, 4, 5, 6, 7, 8, 9]
CLASS_NAMES = {
    0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car',
    4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle',
    8: 'bus', 9: 'motor'
}

# Use a sequence of images to simulate video frames
val_images_path = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-val/images'
frames = sorted(os.listdir(val_images_path))[:20]  # 20 frames

# Color map for tracking IDs
def get_color(track_id):
    np.random.seed(track_id)
    return tuple(np.random.randint(50, 255, 3).tolist())

tracked_frames = []
human_counts_per_frame = []
car_counts_per_frame = []

for frame_file in frames:
    img_path = os.path.join(val_images_path, frame_file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # Run ByteTrack
    track_results = model.track(
        img_path,
        conf=0.25,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )[0]

    human_count = 0
    car_count = 0

    if track_results.boxes.id is not None:
        for box, track_id in zip(track_results.boxes, track_results.boxes.id):
            cls = int(box.cls)
            tid = int(track_id)
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            color = get_color(tid)

            if cls in HUMAN_CLASSES:
                human_count += 1
            elif cls in CAR_CLASSES:
                car_count += 1
            else:
                continue

            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
            label = f"ID:{tid} {CLASS_NAMES[cls]}"
            cv2.putText(img_rgb, label, (x1, max(y1-5, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)

    # Count overlay
    cv2.rectangle(img_rgb, (0, 0), (w, 32), (0, 0, 0), -1)
    cv2.putText(img_rgb, f"Humans: {human_count} | Vehicles: {car_count}",
                (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    tracked_frames.append(img_rgb)
    human_counts_per_frame.append(human_count)
    car_counts_per_frame.append(car_count)

print(f"Tracked {len(tracked_frames)} frames successfully!")

# ── Visualize 6 tracked frames ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Task 04: Object Tracking with ByteTrack\nEach color = unique tracked ID',
             fontsize=14, fontweight='bold')

for ax, frame, h, c in zip(axes.flatten(),
                             tracked_frames[:6],
                             human_counts_per_frame[:6],
                             car_counts_per_frame[:6]):
    ax.imshow(frame)
    ax.set_title(f'Humans: {h} | Vehicles: {c}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/tracking_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Count over frames chart ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(human_counts_per_frame, color='#e74c3c', linewidth=2, marker='o', label='Humans')
ax.plot(car_counts_per_frame, color='#2ecc71', linewidth=2, marker='s', label='Vehicles')
ax.set_title('Human & Vehicle Count Across Frames', fontweight='bold')
ax.set_xlabel('Frame')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/count_over_frames.png', dpi=150, bbox_inches='tight')
plt.show()
print("Tracking visualization saved!")

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import cv2
import os
import numpy as np

# Load best model
model = YOLO(f"{results.save_dir}/weights/best.pt")

# ── 1. Formal Evaluation on Val Set ─────────────────────────────────
print("Running evaluation on validation set...")
metrics = model.val(
    data='/content/visdrone.yaml',
    split='val',
    verbose=False
)

print("\n" + "="*45)
print("       EVALUATION RESULTS")
print("="*45)
print(f"  mAP@50        : {metrics.box.map50:.4f}")
print(f"  mAP@50-95     : {metrics.box.map:.4f}")
print(f"  Precision     : {metrics.box.mp:.4f}")
print(f"  Recall        : {metrics.box.mr:.4f}")
print("="*45)

# ── 2. Per-Class mAP ─────────────────────────────────────────────────
CLASS_NAMES = ['pedestrian','people','bicycle','car','van',
               'truck','tricycle','awning-tricycle','bus','motor']

per_class_ap = metrics.box.ap50

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if i in [0,1] else '#2ecc71' for i in range(10)]
bars = ax.bar(CLASS_NAMES, per_class_ap, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Per-Class AP@50\nRed = Human Classes | Green = Vehicle Classes',
             fontweight='bold', fontsize=13)
ax.set_ylabel('AP@50')
ax.set_ylim(0, 0.6)
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, per_class_ap):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('/content/per_class_ap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Per-class AP chart saved!")

# ── 3. Training Curves ───────────────────────────────────────────────
df = pd.read_csv(f"{results.save_dir}/results.csv")
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('YOLOv8 Training Curves on VisDrone', fontsize=14, fontweight='bold')

metrics_to_plot = [
    ('train/box_loss', 'Box Loss (Train)', '#e74c3c'),
    ('train/cls_loss', 'Class Loss (Train)', '#e67e22'),
    ('train/dfl_loss', 'DFL Loss (Train)', '#9b59b6'),
    ('metrics/mAP50(B)', 'mAP@50', '#2ecc71'),
    ('metrics/precision(B)', 'Precision', '#3498db'),
    ('metrics/recall(B)', 'Recall', '#1abc9c'),
]

for ax, (col, title, color) in zip(axes.flatten(), metrics_to_plot):
    if col in df.columns:
        ax.plot(df[col], color=color, linewidth=2, marker='o', markersize=4)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves_final.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved!")

# ── 4. Final Detection Samples with Count ────────────────────────────
val_images_path = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-val/images'
sample_imgs = sorted(os.listdir(val_images_path))[10:16]

HUMAN_CLASSES = [0, 1]
CAR_CLASSES = [2, 3, 4, 5, 6, 7, 8, 9]
CLS_NAMES = {0:'pedestrian',1:'people',2:'bicycle',3:'car',
             4:'van',5:'truck',6:'tricycle',7:'awning-tricycle',
             8:'bus',9:'motor'}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Task 05: Final Detection Outputs\nRed = Humans | Green = Vehicles',
             fontsize=14, fontweight='bold')

total_humans = 0
total_vehicles = 0

for ax, img_file in zip(axes.flatten(), sample_imgs):
    img_path = os.path.join(val_images_path, img_file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    result = model(img_path, conf=0.25, verbose=False)[0]
    human_count = 0
    car_count = 0

    for box in result.boxes:
        cls = int(box.cls)
        conf = float(box.conf)
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if cls in HUMAN_CLASSES:
            color = (255, 0, 0)
            human_count += 1
        elif cls in CAR_CLASSES:
            color = (0, 200, 0)
            car_count += 1
        else:
            continue

        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_rgb, f"{CLS_NAMES[cls]} {conf:.2f}",
                    (x1, max(y1-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)

    total_humans += human_count
    total_vehicles += car_count

    cv2.rectangle(img_rgb, (0, 0), (w, 32), (0,0,0), -1)
    cv2.putText(img_rgb, f"Humans: {human_count} | Vehicles: {car_count}",
                (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    ax.imshow(img_rgb)
    ax.set_title(f'Humans: {human_count} | Vehicles: {car_count}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/final_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal humans detected across 6 images: {total_humans}")
print(f"Total vehicles detected across 6 images: {total_vehicles}")
print("\nTask 05 Complete ✅")
print("\n🎉 ALL TASKS COMPLETE!")

In [ ]:
from google.colab import files
import os

# Download all output images
output_files = [
    '/content/sample_images.png',
    '/content/class_distribution.png',
    '/content/training_curves.png',
    '/content/detection_results.png',
    '/content/tracking_results.png',
    '/content/count_over_frames.png',
    '/content/per_class_ap.png',
    '/content/final_detection.png',
]

for f in output_files:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded: {f}")
    else:
        print(f"Not found: {f}")